# Slope, depth, roughness and cap attribution

**Finding: slope contains useful missing information, but it is not the sole explanation.** It helps explain 22 versus 41/46; 55 is effectively steeper than 22 and is slow mainly because of prescribed depth and calibrated roughness. No predictive model was refitted here.

## Provenance: what are these slopes?

For all 15 networks, current parent routing slope/head/width/depth values were checked against the hydraulic source frame named by the installed `hydraulic_context_manifest.json`. All values match. The source method is the bounded robust reference-profile workflow:

- Screened RiverSP water-surface observations, acquisition stage offsets, Huber loss and curvature smoothing; smoothing selected by held acquisition cycles.
- Reference profile floor at 5% of static-reference global slope; special SWORD gap anchor in 21 and documented bridge in 46.
- Shared-node heads allocated to units with fixed boundary heads and physical reach lengths. Child interiors follow the existing reconstruction policy. The new segmented blocks use preserved boundary heads divided by their collapsed lengths.

This is **observation-derived water-surface slope with reconstruction assumptions**, not a direct raw measurement of every branch or a bed-slope measurement. The head-allocation method does not fit hydrograph responses, n or the celerity cap. However, reproducing it for an unseen river requires the elevation/profile information and the same processing policy; width/topology alone are insufficient. Old source README files describe the state at their creation; the active installed manifests and numeric comparisons establish current consumption.

Depth follows $h_e=A W_e^{0.65}$ within each network, with a network-specific reference coefficient $A$. That relationship was numerically verified for all parent and segmented edited domains. Depth is a modelling prescription, not an independently observed predictor here. Manning n is calibrated per network. Their mathematical separation below is not a causal claim that these variables are independent.

Source instructions: `RAPID/assets/bounded_robust_slopes_20261002/method.json`, its README, `child_head_reconstruction_20261003/README.md`, and `RAPID/investigations/package_regenerated_hydraulics.py`.

## Geometry-only candidate definitions

Use the edited footprint and the established edge visit probabilities. Define $\mu_L=\sum_e r_e L_e$ and $w_e=r_e L_e/\mu_L$.

- Mean slope: $\bar S=\sum_e w_e S_e$.
- Effective slope emphasizing flat sections: $S_{\mathrm{eff}}=(\sum_e w_e/\sqrt{S_e})^{-2}$.
- Slope-heterogeneity factor: $J_S=\bar S/S_{\mathrm{eff}}\geq1$. Equality holds for uniform slope; flat sections increase it.
- Slope-weighted length proxy: $P_S=\sum_e r_e L_e/\sqrt{S_e}=\mu_L/\sqrt{S_{\mathrm{eff}}}$. We store $P_S/(1\,\mathrm{km})$. Since slope is dimensionless, $P_S$ has length units, **not hours**. It resembles the length-dependent part of travel time only if depth and roughness are held constant and caps are absent.

### Do not add the arithmetic mean-slope ratio

These closed edited regions preserve total boundary head drop. Consequently $\sum r_e L_e S_e=\Delta H_{\mathrm{selected}}$ is unchanged, so

$\bar S_c/\bar S_p=\mu_{L,p}/\mu_{L,c}$.

Verified for all 15 segmented cases to within $4.6\times10^{-13}$. This particular slope ratio duplicates the mean-length ratio. The identity does not automatically apply to an arbitrary partly open edit or a different slope-weighting convention. Effective-slope ratios need not be identical, but may still be strongly redundant.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'RAPID').is_dir())
OUT=ROOT/'RAPID/outputs/hydrograph_closed_unit_extension_20261019/slope_attribution'
d=pd.read_csv(OUT/'slope_descriptors.csv',dtype={'example':str})
z=d[(d.side=='parent')&d.example.isin(['22','41','46','55'])].copy()
z['effective_slope_m_per_km']=z.slope_effective*1000
z['mean_slope_m_per_km']=z.slope_mean*1000
display(z[['example','length_km','mean_slope_m_per_km','effective_slope_m_per_km','slope_heterogeneity','depth_geomean','roughness']].round(5))
display(pd.read_csv(OUT/'provenance.csv',dtype={'example':str}))

,example,length_km,mean_slope_m_per_km,effective_slope_m_per_km,slope_heterogeneity,depth_geomean,roughness
12,22,290.20605,0.24518,0.12259,1.99996,4.74188,0.01472
16,41,264.64850,0.07759,0.07160,1.08362,5.34131,0.02100
20,46,435.61311,0.11179,0.05733,1.94995,2.28663,0.01784
26,55,198.50980,0.18037,0.17643,1.02232,1.44620,0.02748


,example,manifest,source_frame,source_method,source_matches_active,depth_width_exponent_verified
0,02,/Volumes/PhD/river_hierarchy/network_variants/...,/Users/6256481/Code/river-hierarchy/RAPID/asse...,bounded_robust_reference_profile,True,0.65
1,03,/Volumes/PhD/river_hierarchy/network_variants/...,/Users/6256481/Code/river-hierarchy/RAPID/asse...,bounded_robust_reference_profile,True,0.65
2,05,/Volumes/PhD/river_hierarchy/network_variants/...,/Users/6256481/Code/river-hierarchy/RAPID/asse...,bounded_robust_reference_profile,True,0.65
3,13,/Volumes/PhD/river_hierarchy/network_variants/...,/Users/6256481/Code/river-hierarchy/RAPID/asse...,bounded_robust_reference_profile,True,0.65
4,15,/Volumes/PhD/river_hierarchy/network_variants/...,/Users/6256481/Code/river-hierarchy/RAPID/asse...,bounded_robust_reference_profile,True,0.65
5,21,/Volumes/PhD/river_hierarchy/network_variants/...,/Users/6256481/Code/river-hierarchy/RAPID/asse...,bounded_robust_reference_profile,True,0.65
6,22,/Volumes/PhD/river_hierarchy/network_variants/...,/Users/6256481/Code/river-hierarchy/RAPID/asse...,bounded_robust_reference_profile,True,0.65
7,37,/Volumes/PhD/river_hierarchy/network_variants/...,/Users/6256481/Code/river-hierarchy/RAPID/asse...,bounded_robust_reference_profile,True,0.65
8,41,/Volumes/PhD/river_hierarchy/network_variants/...,/Users/6256481/Code/river-hierarchy/RAPID/asse...,bounded_robust_reference_profile,True,0.65
9,45,/Volumes/PhD/river_hierarchy/network_variants/...,/Users/6256481/Code/river-hierarchy/RAPID/asse...,bounded_robust_reference_profile,True,0.65


## Separating the speed factors

The implemented raw celerity is $c=(5/3n)h^{2/3}\sqrt S$, followed by the configured lower/upper caps. Effective speed over the edited domain is $c_{\mathrm{eff}}=1/\sum_e w_e/c_e$.

We evaluated all 16 combinations of actual versus reference slope, depth, roughness and cap. Exact four-factor Shapley attribution of **log effective speed** averages over all replacement orders. It assigns interactions symmetrically and reconstructs the total exactly. This is a deterministic equation diagnostic—not random-forest SHAP, a significance test, or a claim of independent causal effects.

Primary reference: uniform slope $10^{-4}$, depth 5 m, n=0.03, no cap. A second reference (slope $2\times10^{-4}$, depth 10 m, n=0.02) checks reference sensitivity. Within each network, actual geometry/visit weights remain fixed. Differences between network attributions reconstruct their log speed ratio because the uniform reference speed is common.

For the parent edited domains:

| Contrast | Actual effective-speed ratio | Slope log contribution | Depth log contribution | n log contribution | Cap log contribution |
|---|---:|---:|---:|---:|---:|
| 22 / 41 | 1.654 | +0.270 | −0.075 | +0.332 | −0.024 |
| 22 / 46 | 2.737 | +0.380 | +0.481 | +0.169 | −0.023 |
| 22 / 55 | 3.253 | **−0.181** | **+0.783** | **+0.601** | −0.024 |

Positive contributions make 22 faster relative to the comparison network. Contributions add in log space, not m/s or percentages. Under the alternative reference, slope remains positive for 22/41 and 22/46, and negative for 22/55; depth and roughness still dominate the latter contrast. Individual interaction allocations change with reference, so precise percentages should not be treated as physical constants.

The parent cap contribution is relatively small here. This does not contradict the earlier strong effect of the cap on the **parent–child change**: the child in 22 has much larger prescribed depths and more cap-limited length. Both parent and child attributions are saved below.


In [4]:
contrasts=pd.read_csv(OUT/'speed_contrasts.csv')
display(contrasts.round(4))
z=contrasts.query("reference==0 and side=='parent'")
px.bar(z,x='comparison',y=['slope','depth','roughness','cap'],barmode='relative',
       title='Contributions to log effective-speed contrast',labels={'value':'Log-speed contribution'}).show()
attrs=pd.read_csv(OUT/'speed_factor_attributions.csv',dtype={'example':str})
display(attrs[(attrs.reference==0)&attrs.example.isin(['22','41','46','55'])].round(4))

,reference,side,comparison,log_speed_ratio,speed_ratio,slope,depth,roughness,cap
0,0,parent,22/41,0.5032,1.6540,0.2702,-0.0748,0.3315,-0.0237
1,0,parent,22/46,1.0069,2.7371,0.3797,0.4812,0.1690,-0.0230
2,0,parent,22/55,1.1795,3.2526,-0.1807,0.7834,0.6005,-0.0237
3,0,child,22/41,0.2594,1.2961,0.1494,-0.0335,0.2556,-0.1121
4,0,child,22/46,0.9012,2.4625,0.2805,0.6385,0.0934,-0.1112
5,0,child,22/55,1.0371,2.8212,-0.2631,0.8879,0.5246,-0.1121
6,1,parent,22/41,0.5032,1.6540,0.2801,0.0242,0.2905,-0.0917
7,1,parent,22/46,1.0069,2.7371,0.3803,0.5578,0.1421,-0.0733
8,1,parent,22/55,1.1795,3.2526,-0.1632,0.8818,0.5520,-0.0911
9,1,child,22/41,0.2594,1.2961,0.2058,0.0393,0.2338,-0.2195


,example,side,reference,effective_speed,slope,depth,roughness,cap
24,22,parent,0,3.2604,0.1014,-0.0693,0.6883,-0.0237
26,22,child,0,4.1521,0.0505,0.3877,0.6124,-0.1121
32,41,parent,0,1.9712,-0.1688,0.0055,0.3568,0.0000
34,41,child,0,3.2035,-0.0989,0.4212,0.3568,0.0000
40,46,parent,0,1.1912,-0.2783,-0.5506,0.5193,-0.0006
42,46,child,0,1.6862,-0.2300,-0.2508,0.5191,-0.0009
52,55,parent,0,1.0024,0.2821,-0.8528,0.0879,0.0000
54,55,child,0,1.4718,0.3136,-0.5002,0.0879,0.0000


## Does slope help explain the segmented response variation?

Across just these 15 cases, the change in slope-weighted length proxy correlates with centroid shift at Pearson 0.775 / Spearman 0.814. Plain mean-length change gives 0.726 / 0.775. That modest increase supports investigating slope, but is **not** validated predictive improvement or independent evidence after selecting these cases. Parent length alone already correlates strongly with shift, so redundancy and held-network validation remain necessary.

The arithmetic mean hides relevant variability: parent slope-heterogeneity factors are approximately 2.00 in 22 and 1.95 in 46, compared with 1.08 in 41 and 1.02 in 55. This is why an absolute mean slope alone may be insufficient.

**Candidate priority:** start with one absolute parent effective-slope descriptor alongside edited size. Test a slope-heterogeneity descriptor only as a separate addition. Do not add the redundant arithmetic slope ratio. A slope-weighted length proxy is an alternative way to combine existing length with slope, not another independent variable to pile into the same panel automatically.

Before fitting, extend the verified slope descriptors consistently to all original comparison domains, using the agreed footprint convention. The current diagnostic has generated them for the 15 segmented cases only. No slope-augmented prediction score or final predictor decision is claimed.


In [6]:
display(pd.read_csv(OUT/'descriptive_associations.csv').round(4))
features=pd.read_csv(OUT/'segmented_slope_candidates.csv',dtype={'example':str})
px.scatter(features,x='delta_slope_proxy_km',y='centroid_shift_hours',text='example',
           title='Descriptive diagnostic: 15 segmented cases, not held-out predictions').show()

,predictor,pearson,spearman,n
0,parent_length_km,-0.8484,-0.8571,15
1,delta_length_km,0.7255,0.7750,15
2,parent_slope_effective,0.3454,0.7429,15
3,slope_effective_ratio,0.1993,0.2679,15
4,delta_slope_proxy_km,0.7748,0.8143,15


Reproduce: `python RAPID/investigations/closed_unit_extension_20261019/slope_attribution.py` in the project environment. Sources, hashes, reference scenarios and scope are in the output manifest. Focused analytic tests verify uncapped factor contributions and the slope-ratio redundancy. No hydrographs were rerouted and no frozen model or response table was modified.